In [7]:
import cv2
import mediapipe as mp

from glob import glob
import os
from pathlib import Path

from tqdm.notebook import tqdm

import numpy as np

import utils


In [8]:
# Initialize Mediapipe Pose model
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils  
# pose = mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7)


mp_utils = utils.MediapipeUtils(mp_pose, mp_drawing)

In [9]:
# Specify the video file(s)
DATA_DIR = glob("Dataset/*/*/*.mp4")
# DATA_DIR = glob("Dataset/*/*/Augmentation/*.mp4")
DATA_PATH = "MP_Data"

# Landmarks to extract (based on Mediapipe Pose Landmarks)
# landmark_indices = [11, 12, 13, 14, 15, 16]

angle_landmarks = ((12,14),
                   (14,16),
                   (11,13),
                   (13,15))

labels = [i.split("\\")[-1] for i in glob("Dataset\*\*")]

DATA_DIR,labels

(['Dataset\\Adjectives\\Blind\\1-1.mp4',
  'Dataset\\Adjectives\\Blind\\1-2.mp4',
  'Dataset\\Adjectives\\Blind\\1-3.mp4',
  'Dataset\\Adjectives\\Blind\\1-4.mp4',
  'Dataset\\Adjectives\\Blind\\1-5.mp4',
  'Dataset\\Adjectives\\Blind\\1-6.mp4',
  'Dataset\\Adjectives\\Blind\\1-7.mp4',
  'Dataset\\Adjectives\\Blind\\1-8.mp4',
  'Dataset\\Adjectives\\Deaf\\Deaf-1.mp4',
  'Dataset\\Adjectives\\Deaf\\Deaf-2.mp4',
  'Dataset\\Adjectives\\Deaf\\Deaf-3.mp4',
  'Dataset\\Adjectives\\Deaf\\Deaf-4.mp4',
  'Dataset\\Adjectives\\Deaf\\Deaf-5.mp4',
  'Dataset\\Adjectives\\Deaf\\Deaf-6.mp4',
  'Dataset\\Adjectives\\Deaf\\Deaf-7.mp4',
  'Dataset\\Adjectives\\Deaf\\Deaf-8.mp4',
  'Dataset\\Adjectives\\Flat\\Flat-1.mp4',
  'Dataset\\Adjectives\\Flat\\Flat-2.mp4',
  'Dataset\\Adjectives\\Flat\\Flat-3.mp4',
  'Dataset\\Adjectives\\Flat\\Flat-4.mp4',
  'Dataset\\Adjectives\\Flat\\Flat-5.mp4',
  'Dataset\\Adjectives\\Flat\\Flat-6.mp4',
  'Dataset\\Adjectives\\Flat\\Flat-7.mp4',
  'Dataset\\Adjectives\\Fla

In [10]:
try:
    # os.mkdir('MP_data')
    for i in range(len(labels)):
        os.mkdir(f'MP_data/{labels[i]}')
except:
    print("Directory already exists")

Directory already exists


In [11]:
import importlib

importlib.reload(utils)

# Process each video
for video_path in tqdm(DATA_DIR):
    data = []
    cap = cv2.VideoCapture(video_path)
    frame_count = 0

        
    with mp_pose.Pose(min_detection_confidence=0.7,  
                            min_tracking_confidence=0.7) as pose:
    
        while cap.isOpened():

            ret, frame = cap.read()
            if not ret:
                break
                        
            image, results = mp_utils.mediapipe_detection(frame, pose)

            mp_utils.draw_styled_landmarks(image, results)

            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
    
                features = mp_utils.extract_features(landmarks)
                data.append(features)    
            
            else:
                data.append(np.zeros(45))

        data = np.array(data)
        # print(data.shape)
        
        path = video_path.split("/")
        
        # Getting Current action of the video
        action = video_path.split("\\")[-2]
        # action = video_path.split("\\")[-3]
        
        #Get the sequence number
        sequence = video_path.split("\\")[-1][:-4]
        
        npy_path = os.path.join(DATA_PATH, action, sequence)
        # print(npy_path)
        
        np.save(npy_path, data)
        cap.release()


  0%|          | 0/110 [00:00<?, ?it/s]